<a href="https://colab.research.google.com/github/comitanigiacomo/Comitani_85673A_Stream_Analysis/blob/main/Comitani_85673A_Stream_anysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Setup Environment

!apt-get update -qq && apt-get install -y openjdk-17-jdk-headless -qq > /dev/null

!test -d spark && echo "Skipping" || (wget -q https://dlcdn.apache.org/spark/spark-4.1.1/spark-4.1.1-bin-hadoop3.tgz -O spark.tgz && mkdir spark && tar xvf spark.tgz -C spark --strip-components=1)
%pip install -q pyspark py4j kaggle

import os

current_dir = os.path.abspath(".")

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = os.path.join(current_dir, "spark")
os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME', "xxxxx")
os.environ['KAGGLE_KEY'] = os.getenv('KAGGLE_KEY', "xxxxx")

!test -d nytdataset && echo "Skipping" || kaggle datasets download -d "benjaminawd/new-york-times-articles-comments-2020"
!test -d nytdataset && echo "Skipping" || unzip -q -d nytdataset new-york-times-articles-comments-2020.zip
!rm new-york-times-articles-comments-2020.zip 2> /dev/null

print("Ambiente configurato correttamente")


E: List directory /var/lib/apt/lists/partial is missing. - Acquire (13: Permission denied)
Skipping
Note: you may need to restart the kernel to use updated packages.
Skipping
Skipping
Ambiente configurato correttamente


In [12]:
# Dataset Streaming Interface

# Create a Python generator using yield to simulate a data stream on the dataset.
# I need to implement it like I'm working with a massive dataset, so using a function like pandas.read_csv
# would mean cheating, because all the data would be loaded into RAM.
# My goal is to create a lazy iterator using yield instead of return.

import csv
import glob

def stream_all_comments(cartella_dataset):
    pattern = f"{cartella_dataset}/nyt-comments-part*.csv"
    for filepath in sorted(glob.glob(pattern)):
        with open(filepath, mode='r', encoding='utf-8') as file:
            for row in csv.DictReader(file):
                yield row
            

cartella_dataset = "nytdataset/"

stream = stream_all_comments(cartella_dataset)

#Test

#for i in range(3):
#    line = next(stream) 
#    
#    print(f"User {line['userID']}:")
#    print(f"'{line['commentBody'][:60]}...'")
#    print("-" * 40)
#
#print("\nOK")

In [ ]:
# Implementation 1

# I want to know in real time how many unique users are interacting today on the site, 
# to understand if there is a bot attack or just to monitor the level of engagement, 
# so I can figure out the best time to publish an article for maximum visibility.
# Obviously, I don't want to keep all the user IDs in RAM.

# Flajolet-Martin Algorithm

# NOTE: For better precision, I will calculate the median of the averages.

In [4]:
# Implementation 2

# I want to know if a user is commenting for the first time on an article belonging to a certain section. 
# So I need to check, in the shortest time possible, if the user's ID belongs to the group of users who have already commented in that section.

# Bloom Filter

# NOTE: I need to calculate the optimal dimension of the filter: it depends on the False Positive (FP) rate I want to tolerate.

# I can also implement a test: passing users I know haven't commented, and calculating how many times the filter makes a mistake (false positive), to see if the percentage corresponds to the theoretical formula.

In [5]:
# Experiments & Scalability Evaluation

# Read around 500,000 lines of data.

# Then I will use a standard Python data structure (like set() to count the actual unique users), 
# so I can make a comparison between the exact count and my algorithm's estimates.

# I can generate a graph to visualize my predictions with respect to the actual values over time.